# The alpha dial - geometry and performance

One knob, turned across the whole grid. The receptor vector the boosting head
eventually reads is

$$ z_{prot} \;=\; (1-\alpha)\cdot \mathrm{frozen\_SVD}(\mathrm{ESM}) \;+\; \alpha\cdot \mathrm{graph}(\ldots) $$

with **one-hot receptor nodes everywhere in this grid**, so the protein embedding
reaches the model *nowhere else*. That is what makes $\alpha$ an honest fraction
of structure rather than a mixing weight between two things that both contain ESM:

| $\alpha$ | what the receptor vector is |
|---|---|
| **0** | a frozen rank-$k$ rotation of ESM. **Structure alone.** The graph is multiplied by zero. |
| **1** | the signed graph over one-hot receptors. **Function alone.** No protein embedding enters the model anywhere. |

The notebook answers two questions and nothing else:

1. **Geometry** - where between the two extremes does the receptor cloud actually
   sit at each $\alpha$? Measured as alignment to two reference clouds (raw ESM,
   the train response profile) under three second-order measures of increasing
   strictness: RSA (neighbour order) -> CCA (shared linear subspace) -> Procrustes
   (same shape). All three are reported so that **larger is more aligned**.
2. **Performance** - what the head scores at each $\alpha$, against the two
   references that are not the graph at all: `boost` (XGBoost on raw ESM || molecule)
   and `naive` (the constant train mean).

**Error bars are over the splits.** Each point is the mean of the dataset's own five
repeats - folds 1-5, or the cold-molecule seeds 42-46 on m2or/inductive - with a
Student-$t$ 95% interval. At $n=5$ the usual 1.96 approximation is 29% too narrow,
so it is not used. Extra *model* seeds, if any are on disk, widen the interval;
`coverage()` says how many there were.

**Computation lives in [`scripts/analysis/alpha_grid.py`](../../../scripts/analysis/alpha_grid.py);
the producer is `scripts/modeling/train/run_alpha_gate_sweep.py`.** This notebook
loads, aggregates and draws - it trains nothing and writes nothing back into the grid.

In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)          # the grid's default path is repo-relative

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scripts.analysis import alpha_grid as ag

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
print('ROOT:', ROOT)

In [ ]:
# --- house style -------------------------------------------------------------
# Explicit white ground: a dark Jupyter theme otherwise paints black behind axes
# that carry dark text, and every figure here is meant to survive a copy into a
# paper as-is.
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'savefig.bbox': 'tight', 'savefig.dpi': 160,
    'figure.dpi': 110,
    'axes.grid': True, 'grid.color': '#e6e8eb', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'axes.edgecolor': '#9aa1a9', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'xtick.color': '#4b5563', 'ytick.color': '#4b5563',
    'text.color': '#111827', 'axes.labelcolor': '#374151',
    'legend.frameon': False, 'lines.linewidth': 2.0, 'lines.markersize': 4.0,
})

# Four hues, checked pairwise for colour-vision deficiency against this ground
# (worst all-pairs dE 9.3 deutan / 17.6 normal). The two reference clouds are the
# only pair that ever shares a panel, and they are the two furthest apart.
C = {'esm': '#1f6fb4',           # structure
     'fun': '#c9531f',           # function
     'gate': '#1f6fb4',          # the dial's own curve
     'graph_legacy': '#2a8a5f',
     'boost_full': '#6b7280',
     'naive': '#b9bec6'}
DASH = {'boost_full': (0, (5, 2)), 'graph_legacy': (0, (4, 1.5, 1, 1.5)),
        'naive': (0, (1, 2))}
INK, MUTED = '#111827', '#6b7280'


def panels(n, ncols=3, w=3.1, h=2.4, **kw):
    """A grid sized to the number of panels, with the empty tail switched off."""
    ncols = min(ncols, max(n, 1))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows),
                             squeeze=False, **kw)
    ax = axes.ravel()
    for a in ax[n:]:
        a.axis('off')
    return fig, ax[:n]


def band(ax, x, lo, hi, color, alpha=0.16):
    ax.fill_between(x, lo, hi, color=color, alpha=alpha, linewidth=0)


def alpha_axis(ax, alphas):
    ax.set_xlim(-0.03, 1.03)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xticks(list(alphas), minor=True)
    ax.tick_params(axis='x', which='minor', length=2.5, color='#c7ccd1')


def figlegend(fig, handles, ncol=4, pad_in=0.62):
    """One legend per figure, below it, describing the ENCODING - never a box
    inside a panel repeating the same twelve entries six times over.

    It RESERVES the strip it sits in (a fixed 0.62in, so a tall figure does not get
    a proportionally huge gap) rather than being dropped at a figure-relative y,
    which lands it on top of the bottom row's x labels."""
    fig.tight_layout()
    fig.subplots_adjust(bottom=fig.subplotpars.bottom
                        + pad_in / fig.get_size_inches()[1])
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.004),
               ncol=ncol, fontsize=8.5, handlelength=2.4, columnspacing=1.8)


def note_empty(a, text='not run yet'):
    """A panel with nothing in it yet, KEPT in the grid.

    Turning the axes off instead would close the gap and silently shift every panel
    after it, so a half-finished grid would read as a complete one with different
    rows. This whole sweep is meant to be looked at while it is still running."""
    a.text(0.5, 0.5, text, ha='center', va='center', transform=a.transAxes,
           fontsize=8, color=MUTED)
    a.set_xticks([]); a.set_yticks([]); a.grid(False)


def savefig(fig, name):
    if not SAVE_FIGS:
        return
    d = pathlib.Path(FIG_DIR); d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f'{name}.png')
    print('wrote', d / f'{name}.png')

## Knobs

Everything below reads these and nothing else. `MOL_SOURCE = None` keeps both
molecular sources as **separate series**, which doubles every panel and is the way
to ask whether a conclusion depends on the molecule embedding. Same for `SEED`:
`None` pools every model seed on disk into a wider interval, an integer pins one.

In [ ]:
# ============================== KNOBS ========================================
ROOT_DIR   = 'results/graph/v8_alpha_gate'
MOL_SOURCE = 'chemberta'   # 'chemberta' | 'gin' | None -> both, as separate series
NODES      = 'onehot'      # 'onehot' is the separated design; None -> whatever is on disk
DATASET    = None          # None -> all; or ['cc', 'hc'] / 'm2or' / ...
REGIME     = None          # None -> both; or 'transductive' / 'inductive'
SEED       = None          # model seed; None -> every one present (widens the bar)

CI_LEVEL   = 0.95          # Student-t, over the splits
GEOM_SCALE = 'value'       # 'value' = the measure itself | 'z' = against its own null
METRICS    = None          # None -> a headline subset per task; 'all' -> the whole battery
LEVELS     = ('boost_full', 'graph_legacy')   # drawn as lines; see the note below

SAVE_FIGS  = False
FIG_DIR    = 'results/graph/v8_alpha_gate/figs'
# =============================================================================

df = ag.load(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
             dataset=DATASET, regime=REGIME, seed=SEED)

SERIES = sorted(df.series.unique())
ALPHAS = sorted(df.loc[df.arm == 'gate', 'alpha'].dropna().unique())
MOF    = ag.metric_for(df)          # metric of record per dataset
DS_OF  = dict(zip(df.series, df.dataset))

pin = ', '.join(f'{k}={v}' for k, v in df.attrs['pinned'].items()) or '-'
print(f'{len(SERIES)} series, {len(ALPHAS)} alphas, pinned: {pin}')
print('alphas:', ', '.join(f'{a:g}' for a in ALPHAS))

## Read this before any plot

**`coverage`** - a curve drawn over a ragged grid (one $\alpha$ finished on two
folds, the rest on five) is a curve whose wiggles are the *schedule*, and nothing
in a mean $\pm$ CI will say so. `ragged` is that condition as a flag.

**`anchor_check`** - at $\alpha=0$ the receptor vector is a frozen rotation of the
very ESM that `boost` reads raw, so the two are the same information through two
different readers and must nearly agree. A gap wider than the tolerance means the
anchor is wrong - the rank truncation, the normalisation, or the branch itself -
and **every $\alpha$ above it inherits the fault**. This is the grid's own smoke
test, not a result.

In [ ]:
cov = ag.coverage(df)
display(cov.style.hide(axis='index').format(precision=3))
if cov.ragged.any():
    print('!! RAGGED: some alphas rest on fewer splits than others -- the wiggles '
          'below are partly the run schedule, not the dial')

anc = ag.anchor_check(df)
display(anc.style.hide(axis='index').format(precision=3))
bad = anc[~anc.ok] if len(anc) else anc
print('anchor OK on every series' if len(anc) and not len(bad) else
      f'!! anchor off on {len(bad)} series -- read nothing above alpha=0 there'
      if len(bad) else 'no alpha=0 / boost pair on disk yet')

---
# 1. Geometry - where the cloud sits between the two extremes

Two reference clouds, three measures, one $\alpha$ axis. Every measure is oriented
so that **larger means closer**, so the dial working means the ESM curve **falls**
and the profile curve **rises**. That is a *definition* being checked, not a
finding - `audit` below turns it into a number, and it comes before any reading of
the plot.

On the `z` scale (`GEOM_SCALE = 'z'`) each value is expressed against that cell's
own permutation null. It answers whether an alignment is distinguishable from
chance at all, which the raw value cannot - but its size grows with the receptor
count (1237 on M2OR against 24 on HC), so **z is never an effect size across
panels**.

In [ ]:
geo = ag.geometry(df, scale=GEOM_SCALE, level=CI_LEVEL)
aud = ag.audit(geo)
cross = ag.crossover(geo)

chk = aud.pivot_table(index='series', columns=['geom', 'ref'], values='mono')
display(chk.style.format('{:+.2f}').set_caption(
    'mono: Spearman of the mean curve against alpha. Must be near -1 for esm, +1 for fun.'))
off = aud[~aud.ok]
print('every curve travels in the direction the gate defines' if not len(off) else
      f'!! {len(off)} curve(s) travel the WRONG way:')
for r in off.itertuples():
    print(f'   {r.series:<28}{r.geom}/{r.ref}  mono {r.mono:+.2f}  travel {r.travel:.3f}')

### 1.1 Alignment against both extremes, along the dial

One row per series, one column per measure. Where the two curves **cross** is where
the receptor cloud stops being structural and starts being functional.

In [ ]:
ncol = len(ag.GEOMS)
fig, ax = panels(len(SERIES) * ncol, ncols=ncol, w=3.0, h=2.25, sharex=True)

for i, s in enumerate(SERIES):
    for j, g in enumerate(ag.GEOMS):
        a, drawn = ax[i * ncol + j], 0
        for ref in ag.REFS:
            q = geo[(geo.series == s) & (geo.geom == g) & (geo.ref == ref)]
            if q.empty:
                continue
            band(a, q.alpha, q.lo, q.hi, C[ref])
            a.plot(q.alpha, q['mean'], color=C[ref], marker='o')
            drawn += 1
        xc = cross[(cross.series == s) & (cross.geom == g)]
        if len(xc) and np.isfinite(xc.alpha_cross.iloc[0]):
            a.axvline(xc.alpha_cross.iloc[0], color=MUTED, lw=1.0, ls=':')
            a.annotate(f'{xc.alpha_cross.iloc[0]:.2f}',
                       xy=(xc.alpha_cross.iloc[0], 1.0), xycoords=('data', 'axes fraction'),
                       xytext=(3, -10), textcoords='offset points',
                       fontsize=7.5, color=MUTED)
        alpha_axis(a, ALPHAS)
        if not drawn:
            note_empty(a, 'no geometry yet')
        if i == 0:
            a.set_title(ag.GEOM_LABEL[g], pad=8)
        if j == 0:
            a.set_ylabel(s, fontsize=8.5)
        if i == len(SERIES) - 1:
            a.set_xlabel(r'$\alpha$')

unit = 'alignment (higher = closer)' if GEOM_SCALE == 'value' else 'z vs permutation null'
fig.suptitle(f'Receptor cloud vs the two extremes, along the dial - {unit}',
             y=1.005, fontsize=11)
figlegend(fig, [
    Line2D([], [], color=C['esm'], marker='o', label='vs ESM  (structure)'),
    Line2D([], [], color=C['fun'], marker='o', label='vs response profile  (function)'),
    Line2D([], [], color=MUTED, ls=':', lw=1.0, label='crossover (curves scaled within panel)'),
    Line2D([], [], color=MUTED, lw=6, alpha=0.2,
           label=f'band = {CI_LEVEL:.0%} CI over the splits'),
], ncol=4)
savefig(fig, f'geometry_curves_{GEOM_SCALE}')
plt.show()

### 1.2 The crossover, all series at once

The same crossings as vertical lines above, collected. Three markers per row, one
per measure: if they agree, the switch point does not depend on which notion of
"aligned" one happens to prefer. `never` means the two curves do not cross inside
the sampled range - which is itself a reading, not a failure.

In [ ]:
MK = {'rsa': 'o', 'cca': 's', 'procrustes': '^'}
fig, axx = plt.subplots(figsize=(7.2, 0.42 * len(SERIES) + 1.9))
for i, s in enumerate(SERIES):
    q = cross[cross.series == s]
    for r in q.itertuples():
        if np.isfinite(r.alpha_cross):
            axx.scatter(r.alpha_cross, i, marker=MK[r.geom], s=46,
                        color=C['esm'], zorder=3)
        else:
            axx.scatter(1.04, i, marker=MK[r.geom], s=40, facecolors='none',
                        edgecolors=MUTED, zorder=3)
axx.axvline(1.02, color='#d1d5db', lw=0.8)
axx.annotate('never', xy=(1.04, -0.9), fontsize=7.5, color=MUTED, ha='center')
axx.set_yticks(range(len(SERIES))); axx.set_yticklabels(SERIES, fontsize=8.5)
axx.set_ylim(-1.2, len(SERIES) - 0.4); axx.invert_yaxis()
axx.set_xlim(-0.03, 1.09); axx.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
axx.set_xlabel(r'$\alpha$ at which structure and function are equally far along')
axx.grid(axis='y', visible=False)
axx.set_title('Crossover per series and measure', loc='left')
figlegend(fig, [Line2D([], [], ls='', marker=MK[g], color=C['esm'],
                       label=ag.GEOM_LABEL[g].split(' - ')[0]) for g in ag.GEOMS]
          + [Line2D([], [], ls='', marker='o', markerfacecolor='none',
                     color=MUTED, label='no crossing in range')],
          ncol=4)
savefig(fig, 'geometry_crossover')
plt.show()

### 1.3 How far each curve actually travels

The two ends of every curve, $\alpha=0$ on the left of each segment and $\alpha=1$ on
the right. A short segment is a measure the knob barely moved; a segment pointing
the wrong way is the manipulation check failing in the open.

In [ ]:
sub = aud.sort_values(['series', 'geom', 'ref'])
fig, ax = panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)
for i, s in enumerate(SERIES):
    a, q = ax[i], sub[sub.series == s]
    if q.empty:
        note_empty(a, 'no geometry yet'); a.set_title(s, fontsize=9); continue
    labs = []
    for k, r in enumerate(q.itertuples()):
        y = len(q) - 1 - k
        a.plot([r.at0, r.at1], [y, y], color=C[r.ref], lw=2.2, alpha=0.55,
               solid_capstyle='round', zorder=2)
        a.scatter([r.at0], [y], s=26, color=C[r.ref], zorder=3)
        a.scatter([r.at1], [y], s=46, color=C[r.ref], marker='D', zorder=3)
        labs.append(f'{r.geom[:4]} {r.ref}')
    a.set_yticks(range(len(q))); a.set_yticklabels(labs[::-1], fontsize=7.5)
    a.grid(axis='y', visible=False)
    a.set_title(s, fontsize=9)
    a.set_xlabel('alignment' if GEOM_SCALE == 'value' else 'z')
fig.suptitle('From one extreme to the other: the value at alpha=0 and at alpha=1',
             y=1.005, fontsize=11)
figlegend(fig, [
    Line2D([], [], ls='', marker='o', color=INK, label=r'$\alpha=0$  (structure alone)'),
    Line2D([], [], ls='', marker='D', color=INK, label=r'$\alpha=1$  (function alone)'),
    Line2D([], [], color=C['esm'], lw=3, alpha=0.55, label='vs ESM'),
    Line2D([], [], color=C['fun'], lw=3, alpha=0.55, label='vs response profile'),
], ncol=4)
savefig(fig, 'geometry_travel')
plt.show()

---
# 2. Performance - what the head scores along the same dial

Same rows, same $x$. The curve is the pipeline's own head (`train_boost` on
`[ z_prot || raw molecule ]`); the flat lines are the two things that are **not the
graph at all**:

* **`boost`** - XGBoost on `[ raw ESM || molecule ]`. The number a graph has to beat,
  and on some cells still the ceiling.
* **`naive`** - the constant train mean. $R^2$'s honest zero, and AUROC 0.5.
* **`legacy`** - the pre-v8 graph, if it was run. It is $\alpha=1$ up to one global
  scalar applied inside the forward pass: invisible to every geometry readout and
  to a tree head, but not to the gradient, so it is a separate arm and not a synonym.

The metric of record is per dataset - $R^2$ on the continuous insect panels, AUROC
on the binary pool. A notebook that picks one metric for the whole grid leaves the
m2or panels empty.

The reference lines carry **no band**. Their marginal interval is not the
uncertainty of the *comparison*, and drawing it invites the overlap fallacy: two
overlapping intervals read as "no difference" even where the paired difference is
unambiguous - which is the usual case here, since the arms share the split and the
draw. **2.2 is the panel that carries the interval that is actually about the gap.**

`LEVELS` decides which of the three gets a **line**. `naive` is left out by default
and printed as a number instead: it sits at 0 ($R^2$) or 0.5 (AUROC), and a line
there stretches every panel's axis across a range nothing in it occupies, which
flattens the curve being read into a streak at the top. Put `'naive'` back in
`LEVELS` when the distance to chance is the point.

### 2.1 The metric of record along the dial

In [ ]:
fig, ax = panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)
for i, s in enumerate(SERIES):
    a, m = ax[i], MOF[DS_OF[s]]
    one = df[df.series == s]
    cur = ag.curve(one, m, level=CI_LEVEL)
    if cur.empty:
        note_empty(a); a.set_title(s, fontsize=9); continue
    band(a, cur.alpha, cur.lo, cur.hi, C['gate'])
    a.plot(cur.alpha, cur['mean'], color=C['gate'], marker='o', zorder=3)
    lev = ag.levels(one, m, level=CI_LEVEL).set_index('arm')
    for arm in LEVELS:
        if arm not in lev.index:
            continue
        r = lev.loc[arm]
        a.axhline(r['mean'], color=C[arm], ls=DASH[arm], lw=1.4, zorder=2)
    # No band around the reference lines. Their MARGINAL interval is not the
    # uncertainty of the comparison, and drawing it invites the overlap fallacy --
    # two overlapping intervals read as "no difference" when the paired difference
    # is significant, which is exactly the case here (the arms share the split AND
    # the draw). Section 2.2 carries the interval that is actually about the gap.
    # `naive` is ANNOTATED, not drawn: it sits at 0 (R2) or 0.5 (AUROC), and a line
    # there stretches the axis over a range nothing in the panel occupies -- every
    # curve then collapses into a flat streak at the top. The number is the part
    # that matters; the pixels are not.
    if 'naive' in lev.index and 'naive' not in LEVELS:
        a.annotate(f"naive {lev.loc['naive', 'mean']:.3f}", xy=(0.02, 0.04),
                   xycoords='axes fraction', fontsize=7.5, color=MUTED)
    alpha_axis(a, ALPHAS)
    a.set_title(s, fontsize=9)
    a.set_ylabel(m); a.set_xlabel(r'$\alpha$')
fig.suptitle('Prediction along the dial, against the references that are not the graph',
             y=1.005, fontsize=11)
figlegend(fig, [
    Line2D([], [], color=C['gate'], marker='o', label=r'gate (the $\alpha$ curve)'),
] + [Line2D([], [], color=C[a], ls=DASH[a], label=ag.ARM_LABEL[a]) for a in LEVELS]
  + [Line2D([], [], color=MUTED, lw=6, alpha=0.2,
            label=f'band = {CI_LEVEL:.0%} CI over the splits')],
  ncol=2 + len(LEVELS))
savefig(fig, 'performance_curves')
plt.show()

### 2.2 Paired against `boost`, split by split

The same numbers as above, but **differenced cell by cell** before averaging. Both
arms ran on the same split with the same draw, so the difference removes the split
and the draw at once - which is far stronger evidence than two overlapping
intervals, and it is the only form in which a 0.01 effect over five folds is
readable at all.

A filled marker is an $\alpha$ whose interval excludes zero. `won` (printed below)
counts the splits in favour, out of five.

In [ ]:
fig, ax = panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)
for i, s in enumerate(SERIES):
    a, m = ax[i], MOF[DS_OF[s]]
    d = ag.delta_vs(df[df.series == s], m, level=CI_LEVEL)
    if d.empty:
        note_empty(a); a.set_title(s, fontsize=9); continue
    a.axhline(0, color=INK, lw=1.0, zorder=2)
    band(a, d.alpha, d.lo, d.hi, C['gate'])
    a.plot(d.alpha, d['mean'], color=C['gate'], zorder=3)
    sig = (d.lo > 0) | (d.hi < 0)
    a.scatter(d.alpha[sig], d['mean'][sig], s=34, color=C['gate'], zorder=4)
    a.scatter(d.alpha[~sig], d['mean'][~sig], s=30, facecolors='white',
              edgecolors=C['gate'], linewidths=1.3, zorder=4)
    alpha_axis(a, ALPHAS)
    a.set_title(s, fontsize=9)
    a.set_ylabel(f'{m}  (gate - boost)'); a.set_xlabel(r'$\alpha$')
fig.suptitle('Paired difference against boost, on the same splits', y=1.005, fontsize=11)
figlegend(fig, [
    Line2D([], [], color=C['gate'], marker='o', label='interval excludes zero'),
    Line2D([], [], color=C['gate'], marker='o', markerfacecolor='white', label='it does not'),
    Line2D([], [], color=INK, lw=1.0, label='gate = boost'),
    Line2D([], [], color=MUTED, lw=6, alpha=0.2, label=f'band = {CI_LEVEL:.0%} CI of the difference'),
], ncol=4)
savefig(fig, 'performance_delta')
plt.show()

won = pd.concat([ag.delta_vs(df[df.series == s], MOF[DS_OF[s]], level=CI_LEVEL)
                 for s in SERIES], ignore_index=True)
display(won.pivot_table(index='series', columns='alpha', values='won')
        .style.format('{:.0f}').set_caption('splits out of 5 on which the gate beat boost'))

### 2.3 The whole battery

Every metric the task admits, on the same dial. The insects carry the pooled
regression scores **and** a discrete block: the response binarised at `rec0` - above
each receptor's own **train** centre, the same convention the graph's edge signs
use, and fit on train so the label definition never sees the held-out rows.

Set `METRICS = 'all'` in the knobs for the full battery, or a list of your own.

In [ ]:
HEADLINE = {'regression': ['R2', 'Spearman', 'rec0_AUROC', 'rec0_MCC'],
            'classification': ['AUROC', 'AUPRC', 'MCC', 'F1']}

for task in ['regression', 'classification']:
    ser = [s for s in SERIES if ag.TASK[DS_OF[s]] == task]
    if not ser:
        continue
    have = ag.metrics_available(df, DS_OF[ser[0]])
    if METRICS == 'all':
        mets = have
    elif METRICS:
        mets = [m for m in METRICS if m in have]
    else:
        mets = [m for m in HEADLINE[task] if m in have]
    if not mets:
        continue
    ncol = len(mets)
    fig, ax = panels(len(ser) * ncol, ncols=ncol, w=2.5, h=2.0, sharex=True)
    for i, s in enumerate(ser):
        one = df[df.series == s]
        for j, m in enumerate(mets):
            a = ax[i * ncol + j]
            cur = ag.curve(one, m, level=CI_LEVEL)
            if cur.empty:
                note_empty(a)
            else:
                band(a, cur.alpha, cur.lo, cur.hi, C['gate'])
                a.plot(cur.alpha, cur['mean'], color=C['gate'], marker='o', ms=3)
                for r in ag.levels(one, m, arms=LEVELS,
                                   level=CI_LEVEL).itertuples():
                    a.axhline(r.mean, color=C[r.arm], ls=DASH[r.arm], lw=1.2)
                alpha_axis(a, ALPHAS)
            if i == 0:
                a.set_title(m, pad=6)
            if j == 0:
                a.set_ylabel(s, fontsize=8)
            if i == len(ser) - 1:
                a.set_xlabel(r'$\alpha$')
    fig.suptitle(f'{task} panels - every metric on the same dial', y=1.005, fontsize=11)
    figlegend(fig, [
        Line2D([], [], color=C['gate'], marker='o', label='gate'),
    ] + [Line2D([], [], color=C[a], ls=DASH[a], label=ag.ARM_LABEL[a]) for a in LEVELS]
      + [Line2D([], [], color=MUTED, lw=6, alpha=0.2,
                label=f'band = {CI_LEVEL:.0%} CI over the splits')],
      ncol=2 + len(LEVELS))
    savefig(fig, f'performance_battery_{task}')
    plt.show()

### 2.4 The two ends as numbers

The same thing the plots say, for pasting. `scripts/analysis/headline_table.py` is
the canonical scoreboard - it pins seed 42 to stay line-for-line comparable with
the published series, where this notebook defaults to every seed on disk.

In [ ]:
rows = []
for s in SERIES:
    one, m = df[df.series == s], MOF[DS_OF[s]]
    cur = ag.curve(one, m, level=CI_LEVEL).set_index('alpha')
    lev = ag.levels(one, m, level=CI_LEVEL).set_index('arm')
    d1 = ag.delta_vs(one, m, level=CI_LEVEL)
    d1 = d1[np.isclose(d1.alpha, 1.0)]
    cell = lambda t: ('-' if t is None or not np.isfinite(t[0])
                      else f'{t[0]:.3f}' + (f' ±{t[1]:.3f}' if np.isfinite(t[1]) else ''))
    g = lambda fr, k: (None if k not in fr.index else
                       (fr.loc[k, 'mean'], fr.loc[k, 'hw']))
    rows.append({'series': s, 'metric': m,
                 'boost': cell(g(lev, 'boost_full')),
                 'alpha=0': cell(g(cur, 0.0)),
                 'alpha=1': cell(g(cur, 1.0)),
                 'legacy': cell(g(lev, 'graph_legacy')),
                 'naive': cell(g(lev, 'naive')),
                 'a1 - boost': ('-' if d1.empty else
                                f"{d1['mean'].iloc[0]:+.3f}"
                                + (f" ±{d1['hw'].iloc[0]:.3f}" if np.isfinite(d1['hw'].iloc[0]) else '')
                                + f"  [{int(d1['won'].iloc[0])}/{int(d1['n'].iloc[0])}]")})
tab = pd.DataFrame(rows)
display(tab.style.hide(axis='index').set_caption(
    f'mean ± {CI_LEVEL:.0%} CI over the splits; [won/n] counts the splits where the gate beat boost'))

### 2.5 The five numbers behind one point

An interval says how uncertain the mean is; it does not say whether the five splits
*agreed*. On the cold-molecule regimes they routinely do not ($\pm0.1\ R^2$ per
fold), and a reader who never looks at this will over-read a 0.01 difference between
two means that each span 0.2.

In [ ]:
AT = 1.0        # which alpha to open up
for s in SERIES:
    m = MOF[DS_OF[s]]
    v = ag.spread(df[df.series == s], m, alpha=AT)[m].to_numpy()
    if not len(v):
        continue
    print(f'{s:<30}{m:<8}' + '  '.join(f'{x:+.3f}' for x in v)
          + f'   |  mean {v.mean():+.3f}  sd {v.std(ddof=1):.3f}')

---
### What this notebook deliberately does not do

* **It does not retrain anything.** Every number comes from
  `results/graph/v8_alpha_gate/metrics_*.csv`.
* **It does not read the receptor clouds themselves.** The sweep dumps them
  (`dumps/<run>/gate_a<alpha>_f<fold>_s<seed>.npz`, with `z_prot` and `receptors`)
  along with the per-pair predictions, which is what a stratification by receptor
  or by odorant would need. That is a separate question and a separate notebook.
* **It does not read the `fun` geometry on M2OR as binding.** That matrix is sparse,
  so most cells are "not assayed" rather than "no response", and the imputation
  writes the global mean into them. What is left is partly *which pairs were
  measured* - assay design. The `tested mask` control in `mechanism_holdout`
  measured that share at 76-103% of the trained graph's own gain there. On the
  complete insect matrices this does not arise.